# Le Monde 1945 — Text Complexity Metrics

Computes the following metrics for every article in `txt/1945/` and stores them in a single DataFrame:

| Metric | Description |
|---|---|
| `n_tokens` | Total word tokens (alphabetic only) |
| `n_unique_lemmas` | Distinct lemmas (spaCy fr) |
| `ttr` | Type-Token Ratio = unique lemmas / total tokens |
| `mtld` | Measure of Textual Lexical Diversity (length-independent) |
| `hapax_ratio` | Fraction of lemmas that appear exactly once |
| `n_sentences` | Sentence count |
| `mean_sent_len` | Mean words per sentence |
| `std_sent_len` | Std-dev of sentence lengths |
| `mean_word_len` | Mean character length of words |
| `long_word_ratio` | Fraction of words > 7 characters |
| `lix` | LIX readability index |
| `mean_syllables` | Mean syllables per word (pyphen fr) |
| `kandel_moles` | Kandel-Moles French readability score |
| `mean_dep_dist` | Mean dependency distance (syntactic complexity) |
| `noun_phrase_density` | Fraction of tokens that are NOUN / DET / ADJ |

In [1]:
# Install dependencies (run once)
import subprocess, sys

pkgs = ["spacy", "pyphen", "pandas", "numpy", "tqdm"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# Download the French spaCy model if not already present
import spacy
try:
    spacy.load("fr_core_news_md")
    print("fr_core_news_md already installed")
except OSError:
    print("Downloading fr_core_news_md …")
    subprocess.run(
        [sys.executable, "-m", "spacy", "download", "fr_core_news_md", "--quiet"],
        check=True,
    )
    print("Done")

✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')
Done


In [6]:
import math
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyphen
import spacy
from tqdm.auto import tqdm

# ── spaCy model ──────────────────────────────────────────────────────────────
# Uses the medium French model: tok2vec + morphologizer + parser + lemmatizer
nlp = spacy.load("fr_core_news_md")

# ── pyphen French syllabifier ─────────────────────────────────────────────────
dic = pyphen.Pyphen(lang="fr_FR")

print(f"spaCy {spacy.__version__}, model: fr_core_news_md")
print(f"pyphen {pyphen.__version__}")

/Users/msfr/Documents/le_monde/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


spaCy 3.8.13, model: fr_core_news_md
pyphen 0.17.2


## Helper functions

In [7]:
def load_text(path: Path) -> str:
    """Read a Le Monde article txt file and rejoin hyphenated line-breaks."""
    raw = path.read_text(encoding="utf-8")
    # "pa-\nge" → "page"  (line-break hyphenation)
    text = re.sub(r"-\n", "", raw)
    # remaining line breaks → space
    text = text.replace("\n", " ")
    return re.sub(r" {2,}", " ", text).strip()


def parse_file_info(path: Path) -> tuple[str, str]:
    """Return (date, title) from a path like txt/1945/01/01/title_id_id.txt."""
    parts = path.parts
    year, month, day = parts[-4], parts[-3], parts[-2]
    title = path.stem.split("_")[0]
    return f"{year}-{month}-{day}", title


def _mtld(tokens: list[str], threshold: float = 0.72) -> float:
    """
    Measure of Textual Lexical Diversity.
    Bidirectional: average of forward and backward passes.
    Returns NaN for texts shorter than 10 tokens.
    """
    if len(tokens) < 10:
        return float("nan")

    def _one_pass(toks):
        segments = 0
        start = 0
        seen: set[str] = set()
        for i, t in enumerate(toks):
            seen.add(t)
            ttr = len(seen) / (i - start + 1)
            if ttr < threshold:
                segments += 1
                start = i + 1
                seen = set()
        # partial trailing segment
        remaining = len(toks) - start
        if remaining > 0:
            partial_ttr = len(seen) / remaining
            segments += (1 - partial_ttr) / (1 - threshold)
        return len(toks) / segments if segments > 0 else len(toks)

    return (_one_pass(tokens) + _one_pass(tokens[::-1])) / 2


def compute_metrics(path: Path) -> dict | None:
    """
    Load one article, run the spaCy pipeline, and return a dict of all metrics.
    Returns None for empty or un-parseable files.
    """
    date, title = parse_file_info(path)
    text = load_text(path)
    if not text:
        return None

    doc = nlp(text)

    # Alphabetic tokens only (excludes punctuation, numbers, symbols)
    words = [t for t in doc if t.is_alpha]
    if not words:
        return None

    n_tokens = len(words)
    lemmas = [t.lemma_.lower() for t in words]
    lemma_counts = Counter(lemmas)
    n_unique = len(lemma_counts)

    # ── Lexical richness ──────────────────────────────────────────────────────
    ttr = n_unique / n_tokens
    hapax_ratio = sum(1 for c in lemma_counts.values() if c == 1) / n_unique
    mtld = _mtld(lemmas)

    # ── Sentence length ───────────────────────────────────────────────────────
    sent_lens = [
        sum(1 for t in s if t.is_alpha)
        for s in doc.sents
    ]
    sent_lens = [l for l in sent_lens if l > 0]
    n_sents = len(sent_lens)
    mean_sent_len = float(np.mean(sent_lens)) if sent_lens else float("nan")
    std_sent_len  = float(np.std(sent_lens))  if sent_lens else float("nan")

    # ── Word-level surface stats ──────────────────────────────────────────────
    word_lens = [len(t.text) for t in words]
    mean_word_len   = float(np.mean(word_lens))
    long_word_ratio = sum(1 for l in word_lens if l > 7) / n_tokens

    # ── LIX ──────────────────────────────────────────────────────────────────
    long6 = sum(1 for l in word_lens if l > 6)
    lix = (n_tokens / n_sents) + (long6 * 100 / n_tokens) if n_sents else float("nan")

    # ── Syllables (pyphen) ────────────────────────────────────────────────────
    syl_counts = [max(1, len(dic.positions(t.text)) + 1) for t in words]
    mean_syllables = float(np.mean(syl_counts))

    # ── Kandel-Moles (French readability) ────────────────────────────────────
    kandel_moles = 207 - 1.015 * mean_sent_len - 73.6 * mean_syllables

    # ── Mean dependency distance ──────────────────────────────────────────────
    dep_dists = [
        abs(t.i - t.head.i)
        for t in doc
        if t.is_alpha and t.dep_ != "ROOT"
    ]
    mean_dep_dist = float(np.mean(dep_dists)) if dep_dists else float("nan")

    # ── Noun phrase density ───────────────────────────────────────────────────
    np_pos = {"NOUN", "DET", "ADJ"}
    noun_phrase_density = sum(1 for t in words if t.pos_ in np_pos) / n_tokens

    return {
        "date":               date,
        "title":              title,
        "n_tokens":           n_tokens,
        "n_unique_lemmas":    n_unique,
        "ttr":                round(ttr, 4),
        "mtld":               round(mtld, 2) if not math.isnan(mtld) else float("nan"),
        "hapax_ratio":        round(hapax_ratio, 4),
        "n_sentences":        n_sents,
        "mean_sent_len":      round(mean_sent_len, 2),
        "std_sent_len":       round(std_sent_len, 2),
        "mean_word_len":      round(mean_word_len, 3),
        "long_word_ratio":    round(long_word_ratio, 4),
        "lix":                round(lix, 2),
        "mean_syllables":     round(mean_syllables, 3),
        "kandel_moles":       round(kandel_moles, 2),
        "mean_dep_dist":      round(mean_dep_dist, 3),
        "noun_phrase_density": round(noun_phrase_density, 4),
    }

## Smoke test on a single article

In [4]:
sample = Path("txt/1945/01/01/de-la-liberation-a-la-victoire_1854100_1819218.txt")
result = compute_metrics(sample)
for k, v in result.items():
    print(f"  {k:<22} {v}")

  date                   1945-01-01
  title                  de-la-liberation-a-la-victoire
  n_tokens               453
  n_unique_lemmas        214
  ttr                    0.4724
  mtld                   59.81
  hapax_ratio            0.7477
  n_sentences            28
  mean_sent_len          16.18
  std_sent_len           9.94
  mean_word_len          4.658
  long_word_ratio        0.1634
  lix                    40.02
  mean_syllables         1.448
  kandel_moles           84.0
  mean_dep_dist          2.859
  noun_phrase_density    0.3885


## Run on all 1945 articles

In [8]:
corpus_dir = Path("txt/1945")
files = sorted(corpus_dir.rglob("*.txt"))
print(f"{len(files)} articles found")

rows = []
for path in tqdm(files, desc="computing metrics"):
    row = compute_metrics(path)
    if row is not None:
        rows.append(row)

print(f"\n{len(rows)} articles processed successfully")

695 articles found


computing metrics: 100%|██████████| 695/695 [00:33<00:00, 20.97it/s]


695 articles processed successfully


## Results DataFrame

In [9]:
df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Preview
df.head(10)

,date,title,n_tokens,n_unique_lemmas,ttr,mtld,hapax_ratio,n_sentences,mean_sent_len,std_sent_len,mean_word_len,long_word_ratio,lix,mean_syllables,kandel_moles,mean_dep_dist,noun_phrase_density
0,1945-01-01,au-conseil-des-ministres,36,29,0.8056,51.84,0.8966,1,36.00,0.00,4.583,0.1389,63.78,1.333,72.33,2.771,0.4722
1,1945-01-01,les-allemands-n-ont-pas-evacue-la-finlande,64,49,0.7656,76.46,0.8367,6,10.67,2.43,6.469,0.3906,59.10,2.078,43.22,2.155,0.4688
2,1945-01-01,les-americains-atteignent-les-faubourgs-de-roc...,915,380,0.4153,62.90,0.7000,54,16.94,9.89,5.156,0.2393,49.62,1.598,72.20,2.658,0.4208
3,1945-01-01,les-antecedents-du-ministere-de-l-economie-nat...,924,385,0.4167,73.92,0.7117,48,19.25,9.67,5.421,0.2879,57.35,1.692,62.96,2.684,0.5054
4,1945-01-01,les-condamnations-des-cours-de-justice,28,20,0.7143,23.48,0.7000,5,5.60,4.76,5.393,0.2143,37.74,1.679,77.77,1.870,0.6786
5,1945-01-01,les-evenements-de-grece,331,165,0.4985,49.72,0.7152,20,16.55,9.17,4.949,0.1873,48.27,1.565,75.02,2.424,0.4290
6,1945-01-01,les-negociations-de-m-monnet,145,81,0.5586,48.44,0.6914,8,18.12,14.37,5.283,0.2483,51.92,1.669,65.77,2.464,0.4690
7,1945-01-01,les-producteurs-de-cereales-panifiables-ne-ben...,93,57,0.6129,38.94,0.8070,2,46.50,11.50,5.237,0.2903,78.76,1.753,30.80,3.385,0.5054
8,1945-01-01,les-producteurs-de-cereales-panifiables-ne-ben...,93,57,0.6129,38.94,0.8070,2,46.50,11.50,5.237,0.2903,78.76,1.753,30.80,3.385,0.5054
9,1945-01-01,le-voyage-a-londres-de-m-stettinius,83,56,0.6747,63.62,0.7857,4,20.75,11.97,5.229,0.2651,56.89,1.566,70.66,2.418,0.5060


In [10]:
# Statistical summary of all numeric metrics
metric_cols = [c for c in df.columns if c not in ("date", "title")]
df[metric_cols].describe().round(3)

,n_tokens,n_unique_lemmas,ttr,mtld,hapax_ratio,n_sentences,mean_sent_len,std_sent_len,mean_word_len,long_word_ratio,lix,mean_syllables,kandel_moles,mean_dep_dist,noun_phrase_density
count,695.000,695.000,695.000,695.000,695.000,695.000,695.000,695.000,695.000,695.000,695.000,695.000,695.000,695.000,695.000
mean,245.603,119.122,0.600,55.242,0.769,12.942,19.967,11.809,5.142,0.234,52.719,1.634,66.472,2.742,0.463
std,312.053,102.085,0.124,16.304,0.066,18.706,7.850,5.065,0.345,0.053,9.307,0.123,11.961,0.375,0.057
min,18.000,17.000,0.157,16.680,0.578,1.000,4.730,0.000,4.256,0.100,28.670,1.325,-3.250,1.706,0.266
25%,77.000,52.000,0.517,43.850,0.725,4.000,15.000,8.730,4.909,0.200,46.985,1.553,60.075,2.516,0.429
50%,132.000,81.000,0.600,54.220,0.766,7.000,19.000,11.480,5.116,0.229,51.930,1.627,67.670,2.714,0.460
75%,285.500,152.500,0.680,65.130,0.812,15.000,23.410,14.595,5.326,0.260,57.540,1.698,74.615,2.939,0.495
max,3473.000,602.000,0.947,172.570,0.950,262.000,89.000,37.500,6.875,0.475,121.580,2.421,101.360,4.246,0.679


In [11]:
# Top 10 most lexically rich articles (by MTLD)
df.nlargest(10, "mtld")[["date", "title", "n_tokens", "mtld", "ttr", "hapax_ratio"]]

,date,title,n_tokens,mtld,ttr,hapax_ratio
413,1945-01-11,au-mouvement-national-contre-le-racisme,43,172.57,0.9302,0.9500
308,1945-01-07,l-antigone-de-robert-garnier-au-vieux-colombier,250,131.22,0.6760,0.8698
384,1945-01-10,le-times-preconise-la-conciliation-en-grece,113,115.33,0.7257,0.8780
400,1945-01-11,m-von-papen-a-madrid,34,107.89,0.9118,0.9355
262,1945-01-06,la-politique-de-l-u-r-s-s-dans-le-sud-est-euro...,82,104.60,0.7805,0.8438
526,1945-01-14,oaristys-dans-le-metro,400,102.96,0.5750,0.8043
313,1945-01-09,le-gouvernement-de-lublin-nomme-on-ambassadeur...,19,101.08,0.9474,0.9444
669,1945-01-19,fin-de-la-conference-de-hot-springs,73,99.47,0.7945,0.9138
190,1945-01-05,un-conseil-de-regence-yougoslave-serait-biento...,84,98.78,0.7619,0.9062
177,1945-01-04,l-italie-et-la-france,796,98.07,0.4774,0.7500


In [ ]:
# Top 10 syntactically most complex articles (by mean dependency distance)
df.nlargest(10, "mean_dep_dist")[["date", "title", "mean_sent_len", "mean_dep_dist", "lix", "kandel_moles"]]

In [ ]:
# Correlation matrix between all numeric metrics
df[metric_cols].corr().round(2)

In [ ]:
# Save to CSV for use in other scripts
out = Path("txt/metrics_1945.csv")
df.to_csv(out, index=False)
print(f"Saved {len(df)} rows → {out}")